# 00_setup.ipynb — 環境安裝與驗證
**變星測光管線 v1.0  |  設計規格：DESIGN_DECISIONS_v5.md**

本 notebook 執行一次即可。用途：
1. 安裝所有必要 Python 套件
2. 確認 ASTAP（本機）或 astrometry.net API key（Colab）可用
3. 驗證 `observation_config.yaml` 路徑與格式
4. 建立完整的資料夾結構（含 `logs/`）
5. 測試管線模組匯入（含選用模組 period_analysis）
6. 輸出環境摘要報告

---
**執行前請先確認：**
- 本機：`observation_config.yaml` 已放在 `D:/VarStar/pipeline/`
- Colab：Google Drive 已掛載，`VarStar/` 共用雲端硬碟已加入

## Cell 1 — 執行環境偵測

In [1]:
# -*- coding: utf-8 -*-
import sys
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f"執行環境：{'Google Colab' if IS_COLAB else '本機 / JupyterLab'}")
print(f"Python 版本：{sys.version.split()[0]}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CONFIG_PATH = Path(
        '/content/drive/Shareddrives/VarStar/pipeline/observation_config.yaml'
    )
else:
    _candidates = [
        Path(os.environ.get('VARSTAR_CONFIG', '/nonexistent')),
        Path.cwd() / 'observation_config.yaml',
        Path('D:/VarStar/pipeline/observation_config.yaml'),
    ]
    CONFIG_PATH = next((p for p in _candidates if p.exists()), None)

if CONFIG_PATH is None or not CONFIG_PATH.exists():
    print('\n[ERROR] 找不到 observation_config.yaml')
    print('請手動指定，例如：')
    print('  CONFIG_PATH = Path("D:/VarStar/pipeline/observation_config.yaml")')
else:
    print(f'\n設定檔路徑：{CONFIG_PATH}')
    print('✅ 環境偵測完成')

執行環境：本機 / JupyterLab
Python 版本：3.14.3

設定檔路徑：d:\VarStar\pipeline\observation_config.yaml
✅ 環境偵測完成


## Cell 2 — 套件安裝
首次執行時安裝所有必要套件。版本鎖定下限，允許安全性更新。

### 本機部署說明
**本機（命令列）用戶**：若尚未安裝套件，請先在終端機執行下列指令，**只需執行一次**：
```
cd D:\VarStar\pipeline
pip install -r requirements.txt
```
`requirements.txt` 位於 `pipeline/` 目錄，與 `.py` 檔同層。

**Jupyter / Colab 用戶**：直接執行下方 Cell 即可，無需手動 pip。

In [2]:
# -*- coding: utf-8 -*-
# 核心科學套件
%pip install -q "numpy>=1.24"
%pip install -q "pandas>=2.0"
%pip install -q "scipy>=1.10"
%pip install -q "matplotlib>=3.7"
# 天文套件
%pip install -q "astropy>=5.3"
%pip install -q "photutils>=1.9"
# 相機 RAW / EXIF
%pip install -q "rawpy>=0.18"
%pip install -q "exifread>=3.0"
# 工具套件
%pip install -q "tqdm>=4.65"
%pip install -q "pyyaml>=6.0"
%pip install -q "requests>=2.28"
%pip install -q "packaging>=23.0"
print('\n✅ 套件安裝完成')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.

✅ 套件安裝完成



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 3 — 套件版本驗證

In [3]:
# -*- coding: utf-8 -*-
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

# (package_name, minimum_version)
_REQUIREMENTS = [
    ('numpy',      '1.24'),
    ('pandas',     '2.0'),
    ('scipy',      '1.10'),
    ('matplotlib', '3.7'),
    ('astropy',    '5.3'),
    ('photutils',  '1.9'),
    ('rawpy',      '0.18'),
    ('exifread',   '3.0'),
    ('tqdm',       '4.65'),
    ('pyyaml',     '6.0'),
    ('requests',   '2.28'),
]

all_ok = True
print(f"{'套件':<14} {'版本':<12} {'最低要求':<12} 狀態")
print('─' * 52)
for pkg, min_ver in _REQUIREMENTS:
    try:
        ver = version(pkg)
        ok = Version(ver) >= Version(min_ver)
        status = '✓' if ok else '✗ 版本過舊'
        if not ok:
            all_ok = False
    except PackageNotFoundError:
        ver, status, all_ok = '未安裝', '✗ 未安裝', False
    print(f'{pkg:<14} {ver:<12} >={min_ver:<10} {status}')

print()
if all_ok:
    print('✅ 所有套件版本符合要求')
else:
    print('[WARNING] 部分套件不符合要求，請重新執行 Cell 2。')

套件             版本           最低要求         狀態
────────────────────────────────────────────────────
numpy          2.4.3        >=1.24       ✓
pandas         3.0.1        >=2.0        ✓
scipy          1.17.1       >=1.10       ✓
matplotlib     3.10.8       >=3.7        ✓
astropy        7.2.0        >=5.3        ✓
photutils      2.3.0        >=1.9        ✓
rawpy          0.26.1       >=0.18       ✓
exifread       3.5.1        >=3.0        ✓
tqdm           4.67.3       >=4.65       ✓
pyyaml         6.0.3        >=6.0        ✓
requests       2.32.5       >=2.28       ✓

✅ 所有套件版本符合要求


## Cell 4 — observation_config.yaml 驗證
載入設定檔，檢查必要欄位，印出 session 與目標星清單。

In [4]:
# -*- coding: utf-8 -*-
import yaml

_REQUIRED_KEYS = [
    'paths', 'telescopes', 'cameras', 'observatory',
    'obs_sessions', 'targets', 'calibration', 'aperture', 'comparison',
]

try:
    with CONFIG_PATH.open('r', encoding='utf-8') as fh:
        cfg = yaml.safe_load(fh)
    print(f'設定檔載入成功：{CONFIG_PATH}\n')
except FileNotFoundError:
    raise FileNotFoundError(
        f'找不到設定檔：{CONFIG_PATH}\n請重新執行 Cell 1。'
    )
except yaml.YAMLError as exc:
    raise ValueError(f'YAML 格式錯誤：{exc}')

# 必要欄位檢查（observatory / aperture / comparison 為可選，缺少時警告不中斷）
_required_strict = ['paths', 'telescopes', 'cameras', 'obs_sessions', 'calibration']
_required_warn = ['observatory', 'aperture', 'comparison', 'targets']

missing_strict = [k for k in _required_strict if k not in cfg]
missing_warn = [k for k in _required_warn if k not in cfg]

if missing_strict:
    raise KeyError(f'[ERROR] 缺少必要欄位（管線無法執行）：{missing_strict}')
if missing_warn:
    print(f'[WARNING] 缺少建議欄位（部分功能受限）：{missing_warn}')
else:
    print('必要欄位：✅ 全部存在\n')

# obs_sessions — 注意 yaml 中每筆的 targets 是列表（複數）
sessions = cfg.get('obs_sessions', [])
print(f'obs_sessions（共 {len(sessions)} 個）：')
for s in sessions:
    # targets 欄位為列表，以逗號串接顯示
    target_list = s.get('targets', s.get('target', []))
    if isinstance(target_list, str):
        target_list = [target_list]
    targets_str = ', '.join(target_list) if target_list else '（未設定）'
    print(f"  {s.get('date')}  {targets_str:<20}  "
          f"{s.get('telescope')} + {s.get('camera')}")

targets = cfg.get('targets', {})
if targets:
    print(f'\ntargets（共 {len(targets)} 個）：')
    for name, info in targets.items():
        print(f"  {name:<15}  RA={info.get('ra_deg', '?'):.4f}  "
              f"Dec={info.get('dec_deg', '?'):.4f}  "
              f"V≈{info.get('vmag_approx', '?')}  "
              f"P≈{info.get('period_days_approx', '?')} d")

obs = cfg.get('observatory', {})
if obs:
    print(f"\n觀測站：{obs.get('name')}")
    print(f"  緯度 {obs.get('latitude_deg')}°  "
          f"經度 {obs.get('longitude_deg')}°  "
          f"高度 {obs.get('elevation_m')} m")

print('\n✅ 設定檔驗證完成')

設定檔載入成功：d:\VarStar\pipeline\observation_config.yaml

[WARNING] 缺少建議欄位（部分功能受限）：['observatory', 'aperture', 'comparison', 'targets']
obs_sessions（共 2 個）：
  20251122  AlVel, CCAnd, SXPhe   VixenR200SS + Canon6D2
  20251220  V1162Ori              VixenR200SS + Canon6D2

✅ 設定檔驗證完成


## Cell 5 — 資料夾結構建立與差異報告
依 `obs_sessions` 自動建立所有必要目錄（DESIGN_DECISIONS_v5.md §1.1）。
執行後輸出三段報告：
- **新建目錄**：本次新建的目錄
- **yaml 已定義、目錄已存在**：正常，無需處理
- **孤立目錄**：磁碟上存在但 yaml 中無對應目標，請確認是否保留

本 cell 不刪除任何目錄，所有處置由使用者手動決定。

In [ ]:
# -*- coding: utf-8 -*-
import yaml
from pathlib import Path

with CONFIG_PATH.open('r', encoding='utf-8') as fh:
    cfg = yaml.safe_load(fh)

try:
    import google.colab  # noqa: F401
    project_root = Path(cfg['paths']['colab']['project_root'])
except ImportError:
    project_root = Path(cfg['paths']['local']['project_root'])

data_root = project_root / 'data'
pipeline_root = project_root / 'pipeline'
output_root = project_root / 'output'
print(f'Project root：{project_root}')
print(f'Data root   ：{data_root}')
print(f'Output root ：{output_root}\n')

# ── 目錄建立 ─────────────────────────────────────────────────────────────────
created: list[str] = []

# logs / output 基礎目錄
for d in [pipeline_root / 'logs', output_root, output_root / '_pipeline_log']:
    if not d.exists():
        d.mkdir(parents=True, exist_ok=True)
        created.append(str(d))

# share/catalogs
cat_dir = data_root / 'share' / 'catalogs'
if not cat_dir.exists():
    cat_dir.mkdir(parents=True, exist_ok=True)
    created.append(str(cat_dir))

for session in cfg.get('obs_sessions', []):
    date = str(session['date'])
    _date_fmt = f'{date[:4]}-{date[4:6]}-{date[6:8]}'
    telescope = session.get('telescope', '')
    camera = session.get('camera', '')

    # ── share/calibration/{date}_{telescope}_{camera}/ ────────────────────
    _cal_label = f'{date}_{telescope}_{camera}' if telescope and camera else date
    for frame_type in ['dark', 'flat', 'bias', 'masters']:
        d = data_root / 'share' / 'calibration' / _cal_label / frame_type
        if not d.exists():
            d.mkdir(parents=True, exist_ok=True)
            created.append(str(d))

    # ── data/{YYYY-MM-DD}/{GROUP}/ ────────────────────────────────────────
    # 同 session 可能有多個 group，用 set 去重
    target_list = session.get('targets', [])
    if isinstance(target_list, str):
        target_list = [target_list]

    groups_seen = set()
    for target in target_list:
        tgt_cfg = cfg.get('targets', {}).get(target, {})
        group = tgt_cfg.get('group', target)
        if group in groups_seen:
            continue
        groups_seen.add(group)

        field_root = data_root / _date_fmt / group
        for sub in ['raw', 'wcs', 'splits/R', 'splits/G1', 'splits/G2', 'splits/B']:
            d = field_root / sub
            if not d.exists():
                d.mkdir(parents=True, exist_ok=True)
                created.append(str(d))

# ── 報告輸出 ─────────────────────────────────────────────────────────────────
sep = '─' * 55
print(sep)
print('  目錄狀態報告')
print(sep)

if created:
    print(f'\n新建目錄（{len(created)} 個）：')
    for d in created:
        print(f'  {d}')
else:
    print('\n所有目錄已存在，無需新建。')

print(f'\n{sep}')
print('✅ 資料夾結構建立完成')

## Cell 6 — ASTAP 驗證（本機）
確認 ASTAP 執行檔和星表目錄存在。**Colab 使用者跳過，改執行 Cell 7。**

In [ ]:
# -*- coding: utf-8 -*-
import subprocess
import shutil
import yaml
from pathlib import Path

try:
    import google.colab  # noqa: F401
    print('[SKIP] Colab 環境，請改執行 Cell 7。')
    raise SystemExit(0)
except ImportError:
    pass

with CONFIG_PATH.open('r', encoding='utf-8') as fh:
    cfg = yaml.safe_load(fh)

astap_cfg = cfg.get('astrometry', {}).get('astap', {})
exe       = astap_cfg.get('executable', 'astap_cli')
db_path   = astap_cfg.get('db_path', 'C:/Program Files/astap/d80')

# 執行檔
exe_found = shutil.which(exe)
if exe_found:
    print(f'ASTAP 執行檔：{exe_found}')
    try:
        r = subprocess.run(
            [exe_found, '--version'],
            capture_output=True, text=True, timeout=10
        )
        print(f'版本：{(r.stdout or r.stderr).strip().splitlines()[0]}')
        print('ASTAP 執行檔：✅')
    except Exception as exc:
        print(f'[WARNING] --version 失敗：{exc}')
else:
    print(f'[ERROR] 找不到 ASTAP 執行檔：{exe}')
    print('        請安裝 ASTAP 或在 observation_config.yaml 填入完整路徑：')
    print('        astrometry.astap.executable: "C:/Program Files/astap/astap_cli.exe"')

# 星表目錄
db = Path(db_path)
if db.exists():
    n = (len(list(db.glob('*.290'))) + len(list(db.glob('*.dat')))
         + len(list(db.glob('*.1476'))))
    print(f'\n星表目錄：{db}  ({n} 個檔案)')
    if n == 0:
        print('[WARNING] 目錄存在但無星表檔（.290/.dat/.1476）。請確認星表已下載解壓縮。')
    else:
        print('星表目錄：✅')
else:
    print(f'\n[ERROR] 星表目錄不存在：{db}')
    print('        請填入正確路徑：astrometry.astap.db_path')

## Cell 7 — astrometry.net API key 驗證（Colab）
**本機使用者跳過，改執行 Cell 6。**

In [ ]:
# -*- coding: utf-8 -*-
import json
import urllib.parse
import urllib.request
import yaml

try:
    import google.colab  # noqa: F401
except ImportError:
    print('[SKIP] 本機環境，請改執行 Cell 6。')
    raise SystemExit(0)

with CONFIG_PATH.open('r', encoding='utf-8') as fh:
    cfg = yaml.safe_load(fh)

api_key = (
    cfg.get('astrometry', {})
       .get('astrometry_net', {})
       .get('api_key', '')
)

if not api_key:
    print('[ERROR] 缺少 api_key。')
    print('        請在 yaml 填入：astrometry.astrometry_net.api_key: "YOUR_KEY"')
    print('        取得：https://nova.astrometry.net/api_help')
else:
    payload = urllib.parse.urlencode(
        {'request-json': json.dumps({'apikey': api_key})}
    )
    req = urllib.request.Request(
        'https://nova.astrometry.net/api/login',
        data=payload.encode(), method='POST',
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            result = json.loads(resp.read().decode())
        if result.get('status') == 'success':
            print('astrometry.net API key：✅ 有效')
        else:
            print(f'[ERROR] API key 無效。回應：{result}')
            print('        請至 https://nova.astrometry.net/api_help 確認。')
    except Exception as exc:
        print(f'[ERROR] 連線失敗：{exc}')

## Cell 8 — 管線模組匯入測試
測試所有模組是否可正常匯入。`period_analysis` 為進階選用模組，
匯入失敗時輸出警告但不中斷（不影響標準管線執行）。

In [ ]:
# -*- coding: utf-8 -*-
import sys

pipeline_dir = CONFIG_PATH.parent
if str(pipeline_dir) not in sys.path:
    sys.path.insert(0, str(pipeline_dir))
    print(f'已加入 sys.path：{pipeline_dir}')

# (module_name, function_name, required)
# required=False：選用模組，失敗不影響標準管線
_modules = [
    ('Calibration',     'run_calibration',     True),
    ('plate_solve',     'run_plate_solve',      True),
    ('DeBayer_RGGB',    'run_debayer',          True),
    ('period_analysis', 'run_period_analysis',  False),  # 進階選用
]

all_required_ok = True
print()
print(f"{'模組':<22} {'狀態':<6} 備註")
print('─' * 55)
for mod_name, func_name, required in _modules:
    tag = '（必要）' if required else '（選用）'
    try:
        mod = __import__(mod_name)
        assert hasattr(mod, func_name), f'{func_name} 不存在'
        print(f'  {mod_name:<20} ✅     {tag} {func_name} 可用')
    except (ImportError, AssertionError) as exc:
        if required:
            print(f'  {mod_name:<20} ✗      {tag} {exc}')
            all_required_ok = False
        else:
            print(f'  {mod_name:<20} ─      {tag} {exc}')
            print(f'  {"":<20}        （進階週期分析不可用，標準管線不受影響）')

print()
if all_required_ok:
    print('✅ 所有必要模組匯入正常，可執行標準管線。')
else:
    print('[WARNING] 必要模組有失敗項目，請確認 .py 檔案在 pipeline/ 目錄下。')

## Cell 9 — 環境摘要報告

In [ ]:
# -*- coding: utf-8 -*-
import sys
import platform
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

_PKGS = [
    'numpy', 'pandas', 'scipy', 'matplotlib',
    'astropy', 'photutils', 'rawpy', 'exifread', 'tqdm', 'pyyaml',
]

print('=' * 55)
print('  變星測光管線  環境摘要報告')
print(f'  {datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S UTC")}')
print('=' * 55)
print(f'Python     : {sys.version.split()[0]}')
print(f'Platform   : {platform.system()} {platform.release()} ({platform.machine()})')

try:
    import google.colab  # noqa: F401
    print('Environment: Google Colab')
except ImportError:
    print('Environment: Local / JupyterLab')

print('\n套件版本：')
for pkg in _PKGS:
    try:
        ver = version(pkg)
    except PackageNotFoundError:
        ver = '未安裝'
    print(f'  {pkg:<14} {ver}')

print(f'\n設定檔：{CONFIG_PATH}')
print('=' * 55)
print('✅ 環境驗證完成，可開始執行管線。')
print()
print('執行順序：')
print('  1. python run_pipeline.py --config observation_config.yaml')
print('     （步驟 1–3：校正 → 解算 → 拆色）')
print('  2. 開啟 Photometry.ipynb 執行測光分析（步驟 4）')
print()
print('進階選用（測光完成後）：')
print('  3. python run_pipeline.py --steps period_analysis')
print('     （步驟 5：Lomb-Scargle + DFT + 預白化週期分析）')